# 01 – Preprocessing & features

Loads the raw Finland aFRR market + weather data, cleans it, and builds the lag and calendar features used in the modelling notebooks.

- Input: `data/extended_data_v2.csv` (on Kaggle it's picked up from the input dir instead)
- Output: `processed/cleaned_raw.csv` and `processed/features_ready.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_PATH = Path("data/extended_data_v2.csv")
KAGGLE_PATH = Path("/kaggle/input/finland-afrr-energy-market-and-weather-data/extended_data_v2.csv")

PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_CLEAN = PROCESSED_DIR / "cleaned_raw.csv"
OUT_FEATURES = PROCESSED_DIR / "features_ready.csv"

TARGET = "Up"
LAGS = [1, 2, 3, 6, 12, 24]
MAX_FILL_HOURS = 3  # longest gap we'll forward-fill; anything longer stays NaN

## Load

In [ ]:
src = KAGGLE_PATH if KAGGLE_PATH.exists() else RAW_PATH
df = pd.read_csv(src)
print(f"{src}: {df.shape[0]:,} rows x {df.shape[1]} cols")
df.head()

## First look

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
na = df.isna().sum()
na[na > 0].sort_values(ascending=False)

## Cleaning

Timestamps go to UTC and duplicates are dropped. Text columns get coerced to numbers (with True/False handled so holiday flags don't turn into NaN). Then the data is reindexed to a full hourly range and short gaps are forward-filled.

No back-fill here on purpose: it copies future values into earlier rows, which leaks into the lag features. Reindexed rows are flagged with `is_imputed_row` so they can be excluded later if needed.

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
df = df.dropna(subset=["datetime"]).set_index("datetime").sort_index()

n_dupes = df.index.duplicated().sum()
if n_dupes:
    print(f"Dropping {n_dupes} duplicate timestamps (keeping first)")
    df = df[~df.index.duplicated(keep="first")]

# object columns are usually numbers stored as text
for c in df.select_dtypes(include=["object", "string"]).columns:
    s = df[c].replace({"True": 1, "False": 0, "true": 1, "false": 0})
    df[c] = pd.to_numeric(s, errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan)

full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h", name="datetime")
imputed = ~full_idx.isin(df.index)
print(f"Missing hourly timestamps: {imputed.sum()}")

df = df.reindex(full_idx)
df["is_imputed_row"] = imputed.astype(int)
df = df.ffill(limit=MAX_FILL_HOURS)

print(f"Rows after reindex: {len(df):,}")
print(f"Cells still missing after fill: {int(df.isna().sum().sum()):,}")

df.to_csv(OUT_CLEAN)

## Features

- `Up_next_hour`: the target, i.e. `Up` shifted back one hour
- Lags of `Up` and the main drivers at 1, 2, 3, 6, 12 and 24 hours
- Calendar features. Hour gets a sin/cos encoding so 23:00 and 00:00 sit next to each other.

In [ ]:
if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' not found")

df["Up_next_hour"] = df[TARGET].shift(-1)

lag_candidates = [
    TARGET, "electricity_consumption", "electricity_consumption_forecast",
    "sp", "Up_Cap", "Down_Cap", "air_temperature", "wind_speed",
]
lag_cols = [c for c in lag_candidates if c in df.columns]
skipped = sorted(set(lag_candidates) - set(lag_cols))
if skipped:
    print("Not in data, skipping lags for:", skipped)

lag_feats = pd.DataFrame(
    {f"{c}_lag_{k}": df[c].shift(k) for c in lag_cols for k in LAGS},
    index=df.index,
)
df = pd.concat([df, lag_feats], axis=1)

df["hour"] = df.index.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["day_of_week"] = df.index.weekday
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["month"] = df.index.month

if "is_public_holiday" in df.columns:
    df["is_public_holiday"] = df["is_public_holiday"].fillna(0).astype(int)

In [ ]:
# need the target and every lag present, not just lag 1
n_before = len(df)
df = df.dropna(subset=["Up_next_hour", *lag_feats.columns])
print(f"Dropped {n_before - len(df):,} rows with missing target/lags, {len(df):,} left")

## Save

In [ ]:
df.to_csv(OUT_FEATURES)
print(f"Saved {df.shape[0]:,} x {df.shape[1]} to {OUT_FEATURES}")